# PV Module Degradation and End-of-Life Waste Forecasting

**Type:** Original supplementary research-practice example  
**Domain:** Solar photovoltaic asset management and circular economy

This project links data science methods to PV asset management: Weibull lifetime modelling, annual degradation, cohort analysis and end-of-life waste forecasting. Every dataset below is simulated. The notebook is not a thesis result, not an estimate of Bangladesh's PV waste, and not a substitute for verified field data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import weibull_min

rng=np.random.default_rng(2050)
# Simulated historical retired-module lifetime observations.
true_shape=3.4
true_scale=29.0
retired_lifetimes=weibull_min.rvs(
    c=true_shape,scale=true_scale,loc=0,size=1800,random_state=rng
)
shape_hat,loc_hat,scale_hat=weibull_min.fit(retired_lifetimes,floc=0)
print("Fitted Weibull shape:",round(shape_hat,3))
print("Fitted Weibull scale (years):",round(scale_hat,3))

ages=np.linspace(0,45,300)
survival=weibull_min.sf(ages,shape_hat,loc=0,scale=scale_hat)
hazard=weibull_min.pdf(ages,shape_hat,loc=0,scale=scale_hat)/np.maximum(survival,1e-12)
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].plot(ages,survival)
ax[0].set(title="Weibull survival function",xlabel="Module age (years)",ylabel="Probability still operating")
ax[1].plot(ages,hazard)
ax[1].set(title="Weibull hazard rate",xlabel="Module age (years)",ylabel="Instantaneous hazard")
for a in ax:a.grid(alpha=.2)
plt.tight_layout()
plt.show()

## 1. Simulate a current installed fleet

Create synthetic PV installations by commissioning cohort. Each record has an initial DC capacity, module mass intensity and degradation rate. The Weibull survival function estimates the expected active fleet at the base year.

In [ ]:
cohorts=np.arange(2005,2026)
cohort_rows=[]
for year in cohorts:
    installed_kw=int(rng.integers(2500,9000))
    module_mass_kg_per_kw=float(rng.normal(68,4))
    annual_degradation=float(np.clip(rng.normal(.006,.0012),.002,.011))
    age=2025-year
    survival_prob=weibull_min.sf(age,shape_hat,loc=0,scale=scale_hat)
    active_kw=installed_kw*survival_prob
    current_output_kw=active_kw*(1-annual_degradation)**age
    cohort_rows.append({
        "commissioning_year":year,
        "initial_capacity_kw":installed_kw,
        "module_mass_kg_per_kw":module_mass_kg_per_kw,
        "annual_degradation":annual_degradation,
        "age_2025":age,
        "survival_probability_2025":survival_prob,
        "expected_active_capacity_kw":active_kw,
        "estimated_current_output_kw":current_output_kw
    })
fleet=pd.DataFrame(cohort_rows)
display(fleet.head())
print("Initial installed capacity MW:",round(fleet.initial_capacity_kw.sum()/1000,2))
print("Expected active capacity MW:",round(fleet.expected_active_capacity_kw.sum()/1000,2))
print("Estimated current output MW:",round(fleet.estimated_current_output_kw.sum()/1000,2))

## 2. Estimate annual end-of-life capacity and mass

For each surviving cohort, calculate the conditional probability of failure in each future annual interval. Expected waste is the expected failed capacity multiplied by an assumed module mass per kW. This is an aggregate expectation, not an asset-level prediction.

In [ ]:
base_year=2025
forecast_years=np.arange(2026,2046)
forecast_rows=[]
for forecast_year in forecast_years:
    total_failed_kw=0.0
    total_waste_kg=0.0
    for row in fleet.itertuples():
        previous_age=base_year-row.commissioning_year
        next_age=forecast_year-row.commissioning_year
        s_previous=weibull_min.sf(previous_age,shape_hat,loc=0,scale=scale_hat)
        f_previous=weibull_min.cdf(previous_age,shape_hat,loc=0,scale=scale_hat)
        f_next=weibull_min.cdf(next_age,shape_hat,loc=0,scale=scale_hat)
        conditional_failure=np.clip((f_next-f_previous)/max(s_previous,1e-12),0,1)
        active_kw=row.expected_active_capacity_kw
        failed_kw=active_kw*conditional_failure
        total_failed_kw+=failed_kw
        total_waste_kg+=failed_kw*row.module_mass_kg_per_kw
    forecast_rows.append({
        "year":forecast_year,
        "expected_failed_capacity_kw":total_failed_kw,
        "expected_pv_waste_tonnes":total_waste_kg/1000
    })
forecast=pd.DataFrame(forecast_rows)
display(forecast.head(10).round(2))

## 3. Visualise cohort capacity and projected waste

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(13,4.5))
ax[0].bar(fleet["commissioning_year"],fleet["expected_active_capacity_kw"]/1000)
ax[0].set(title="Expected active capacity by cohort (2025)",xlabel="Commissioning year",ylabel="MW")
ax[1].bar(forecast["year"],forecast["expected_pv_waste_tonnes"])
ax[1].set(title="Expected PV module waste by year",xlabel="Year",ylabel="Tonnes")
for a in ax:a.grid(axis="y",alpha=.2)
plt.tight_layout()
plt.show()

## 4. Sensitivity analysis: lifetime scale

A key uncertainty is module lifetime. Test how waste projections change if the Weibull scale is shorter or longer while holding shape constant. This is a scenario analysis, not a confidence interval.

In [ ]:
def project_waste(scale_value):
    annual=[]
    for forecast_year in forecast_years:
        total_kg=0.0
        for row in fleet.itertuples():
            previous_age=base_year-row.commissioning_year
            next_age=forecast_year-row.commissioning_year
            s_prev=weibull_min.sf(previous_age,shape_hat,loc=0,scale=scale_value)
            f_prev=weibull_min.cdf(previous_age,shape_hat,loc=0,scale=scale_value)
            f_next=weibull_min.cdf(next_age,shape_hat,loc=0,scale=scale_value)
            p=np.clip((f_next-f_prev)/max(s_prev,1e-12),0,1)
            total_kg+=row.expected_active_capacity_kw*p*row.module_mass_kg_per_kw
        annual.append(total_kg/1000)
    return np.asarray(annual)

scenarios={
    "Shorter lifetime (scale -10%)":project_waste(scale_hat*.9),
    "Base fitted lifetime":project_waste(scale_hat),
    "Longer lifetime (scale +10%)":project_waste(scale_hat*1.1)
}
fig,ax=plt.subplots(figsize=(10,4.5))
for name,values in scenarios.items():
    ax.plot(forecast_years,values,marker="o",markersize=3,label=name)
ax.set(title="Lifetime sensitivity of PV waste forecast",xlabel="Year",ylabel="Expected waste (tonnes)")
ax.legend()
ax.grid(alpha=.2)
plt.tight_layout()
plt.show()

## 5. Research interpretation and data requirements

A defensible empirical study would require, at minimum:

- Installed capacity by commissioning year and technology/module type.
- Observed failures, replacements, and right-censoring dates.
- Module mass and material composition by product class.
- Degradation measurements or validated performance ratios.
- Repowering, early replacement, export and reuse pathways.
- Uncertainty intervals and scenario assumptions.
- A documented distinction between technical end-of-life and actual waste generation.

### Questions

1. Why is a survival function more informative than a single assumed lifespan?
2. What bias arises if only retired modules are used and surviving modules are ignored?
3. How would right-censoring be handled in a formal survival model?
4. Which parameters dominate the projected waste volume?
5. How could collection, repair and refurbishment delay waste generation?

**Important:** all figures and values in this notebook are synthetic, for method demonstration only. Do not cite them as Bangladesh PV waste estimates or thesis findings.